# Étape 2 — Mandatory Baselines
## Random Walk · Historical Mean · ARIMA · Random Forest

**Per `prompt.md` RULE 8:** DL is FORBIDDEN until these 4 baselines are evaluated.  
**Source:** `outputs/etape1/splits/masi_{train,val,test}.csv`  
**Pipeline:** `scripts/02_baselines.py`

## 1. Run the pipeline

In [ ]:
import sys, os, json
from pathlib import Path
import importlib.util as _ilu

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image, display

_spec = _ilu.spec_from_file_location('e2', PROJECT_ROOT / 'scripts' / '02_baselines.py')
e2 = _ilu.module_from_spec(_spec)
sys.modules['e2'] = e2
_spec.loader.exec_module(e2)
run_all = e2.run_all

%matplotlib inline
plt.rcParams['figure.dpi'] = 100
pd.set_option('display.max_columns', 12)

In [ ]:
artifacts = run_all(verbose=True)

## 2. Metrics summary (VAL & TEST)

In [ ]:
metrics = artifacts['metrics']
for part in ('val', 'test'):
    rows = []
    for name, m in metrics[part].items():
        rows.append({
            'baseline': name,
            'RMSE': round(m['rmse'], 6),
            'RMSE_CI95': f"[{m['rmse_ci'][0]:.4f}, {m['rmse_ci'][1]:.4f}]",
            'MAE': round(m['mae'], 6),
            'DA': round(m['directional_accuracy'], 4),
            'DA_CI95': f"[{m['da_ci'][0]:.4f}, {m['da_ci'][1]:.4f}]" if m['da_ci'] else '—',
            'Sharpe': round(m['sharpe_annualized'], 3),
            'MDD': round(m['max_drawdown'], 3),
            'Trades': m['n_trades'],
        })
    print(f'\n=== {part.upper()} ===')
    display(pd.DataFrame(rows))

## 3. Predictions plot — VAL set

In [ ]:
Image(artifacts['plot_paths'][0])

## 4. Predictions plot — TEST set (the out-of-sample result)

In [ ]:
Image(artifacts['plot_paths'][1])

## 5. Bar chart summary across baselines

In [ ]:
Image(artifacts['plot_paths'][2])

## 6. Random Forest feature importance

In [ ]:
imp = artifacts['rf_importance']
imp_sorted = sorted(imp.items(), key=lambda x: -x[1])
imp_df = pd.DataFrame(imp_sorted, columns=['feature', 'importance'])
display(imp_df)

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh([f for f, _ in imp_sorted], [v for _, v in imp_sorted], color='#2ca02c')
ax.set_xlabel('Importance')
ax.set_title('Random Forest — Feature Importance')
ax.invert_yaxis()
ax.grid(alpha=0.3, axis='x')
plt.tight_layout(); plt.show()

## 7. ARIMA model info

In [ ]:
print(f"Selected order: ARIMA{tuple(artifacts['arima_info']['order'])}")
print(f"AIC: {artifacts['arima_info']['aic']:.2f}")

## 8. Key takeaways for Étape 3 + 5

| Insight | Implication |
|---------|-------------|
| Random Forest DA on TEST = 53.3% [95% CI 0.500, 0.564] | **Marginal statistical significance** — directional skill confirmed |
| ARIMA(2,0,2) Sharpe on TEST = 1.168 with 414 trades | Strong point estimate; **threshold for CNN-LSTM** to beat in Étape 5 |
| Top RF features: `log_return`, `eur_mad`, `gold_close`, `brent_close`, `lhm_close` | **Multi-factor design validated** empirically (Étape 0 v2 decision was correct) |
| RW/HM have lowest RMSE but no skill (DA ≈ 50% or worse) | **RMSE alone is misleading** — Sharpe + DA are the real metrics |
| Bootstrap CI lower bounds for DA hover around 0.50 | **Honest reporting**: results are marginally significant, not overwhelming |

**Performance floor for Étape 5 (CNN-LSTM):**
- DA ≥ 0.55 on TEST  (+3 pp over RF)
- Sharpe ≥ 1.30 on TEST (+10% over ARIMA)
- MDD no worse than −0.22

**If CNN-LSTM fails these → ship Random Forest** per `prompt.md` RULE 8.